In [ ]:
# Полный код сканера ПДн с отчетами в CSV, JSON и Markdown

from __future__ import annotations
import os
import re
import io
import json
import csv
import itertools
import mimetypes
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Iterator
from datetime import datetime
from dataclasses import dataclass, field, asdict
import argparse
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from collections import Counter

# ============================================================================
# Конфигурации и константы
# ============================================================================

INCLUDE_EXTS = {'doc', 'docx', 'gif', 'html', 'ipynb', 'jpeg', 'jpg', 'pdf', 
                'php', 'png', 'rtf', 'xls', 'xlsx', 'txt', 'csv', 'json', 'tif', 'tiff'}

PDN_CATEGORIES = ['обычные', 'государственные', 'платёжные', 'биометрические', 'специальные']

# ============================================================================
# Вспомогательные функции
# ============================================================================

def detect_encoding(raw_bytes: bytes) -> str:
    try:
        import chardet
        res = chardet.detect(raw_bytes)
        return res.get('encoding') or 'utf-8'
    except:
        return 'utf-8'

def luhn_check(number: str) -> bool:
    digits = [int(d) for d in re.sub(r'\D', '', number)]
    if not (13 <= len(digits) <= 19):
        return False
    s = 0
    parity = len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9:
                d -= 9
        s += d
    return s % 10 == 0

def snils_valid(snils: str) -> bool:
    nums = re.sub(r'\D', '', snils)
    if len(nums) != 11:
        return False
    base = [int(x) for x in nums[:9]]
    check = int(nums[9:])
    s = sum((9 - i) * d for i, d in enumerate(base))
    if s < 100:
        c = s
    elif s in (100, 101):
        c = 0
    else:
        c = s % 101
        if c == 100:
            c = 0
    return c == check

def inn_valid(inn: str) -> bool:
    nums = re.sub(r'\D', '', inn)
    if len(nums) == 10:
        w = [2, 4, 10, 3, 5, 9, 4, 6, 8]
        c = sum(int(nums[i]) * w[i] for i in range(9)) % 11 % 10
        return c == int(nums[9])
    elif len(nums) == 12:
        w1 = [7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        w2 = [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        c1 = sum(int(nums[i]) * w1[i] for i in range(11)) % 11 % 10
        c2 = sum(int(nums[i]) * w2[i] for i in range(11)) % 11 % 10
        return c1 == int(nums[10]) and c2 == int(nums[11])
    return False

def has_context(text: str, idx: int, window: int, *keywords: str) -> bool:
    start = max(0, idx - window)
    end = min(len(text), idx + window)
    chunk = text[start:end].lower()
    return any(k.lower() in chunk for k in keywords)

# ============================================================================
# Извлечение текста из файлов
# ============================================================================

def extract_text_from_bytes(content: bytes, ext: str) -> str:
    try:
        if ext in {'txt', 'csv', 'json', 'log', 'md', 'py', 'js', 'html', 'xml', 'php', 'rtf'}:
            for enc in ['utf-8', 'cp1251', 'latin-1']:
                try:
                    return content.decode(enc, errors='ignore')
                except:
                    continue
            return content.decode('utf-8', errors='ignore')
        
        elif ext == 'pdf':
            try:
                import PyPDF2
                import io
                reader = PyPDF2.PdfReader(io.BytesIO(content))
                text = ''
                for page in reader.pages:
                    try:
                        page_text = page.extract_text()
                        if page_text:
                            text += page_text + '\n'
                    except:
                        pass
                if text.strip():
                    return text
            except:
                pass
            
            try:
                from pdfminer.high_level import extract_text as pdfminer_extract
                import io
                text = pdfminer_extract(io.BytesIO(content)) or ''
                if text.strip():
                    return text
            except:
                pass
            
            return ''
        
        elif ext in {'xls', 'xlsx', 'xlsm'}:
            try:
                import pandas as pd
                import io
                df = pd.read_excel(io.BytesIO(content), header=None, dtype=str)
                return ' '.join(df.stack().astype(str).tolist())
            except:
                return ''
        
        elif ext == 'docx':
            try:
                import docx
                import io
                doc = docx.Document(io.BytesIO(content))
                parts = []
                for p in doc.paragraphs:
                    if p.text:
                        parts.append(p.text)
                for tbl in doc.tables:
                    for row in tbl.rows:
                        row_text = ' \t '.join(cell.text for cell in row.cells if cell.text)
                        if row_text:
                            parts.append(row_text)
                return '\n'.join(parts)
            except:
                return ''
        
        elif ext == 'doc':
            try:
                text = content.decode('latin-1', errors='ignore')
                text = re.sub(r'[^\x20-\x7E\u0400-\u04FF]+', ' ', text)
                return re.sub(r'\s+', ' ', text)
            except:
                return ''
        
        elif ext == 'html':
            try:
                txt = content.decode('utf-8', errors='ignore')
                from bs4 import BeautifulSoup
                soup = BeautifulSoup(txt, 'html.parser')
                return soup.get_text(' ')
            except:
                return content.decode('utf-8', errors='ignore')
        
        elif ext == 'ipynb':
            try:
                data = json.loads(content.decode('utf-8', errors='ignore'))
                parts = []
                for cell in data.get('cells', []):
                    src = cell.get('source', [])
                    if isinstance(src, list):
                        parts.append(''.join(src))
                    elif isinstance(src, str):
                        parts.append(src)
                return '\n'.join(parts)
            except:
                return ''
        
        elif ext in {'jpg', 'jpeg', 'png', 'gif', 'tif', 'tiff', 'bmp'}:
            try:
                import pytesseract
                from PIL import Image
                import io
                img = Image.open(io.BytesIO(content))
                if img.mode not in ('RGB', 'L'):
                    img = img.convert('RGB')
                if img.width * img.height > 4000000:
                    img.thumbnail((2000, 2000))
                return pytesseract.image_to_string(img, lang='rus+eng')
            except:
                return ''
        
        else:
            return content.decode('utf-8', errors='ignore')
    except Exception as e:
        return ''

# ============================================================================
# Детекторы ПДн
# ============================================================================

EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?:(?:\+7|8)\s*\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2})")
FIO_RE = re.compile(r"\b[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+)?\b")
DOB_RE = re.compile(r"\b(\d{2}[./]\d{2}[./]\d{4})\b")
INDEX_RE = re.compile(r"\b\d{6}\b")
SNILS_RE = re.compile(r"\b\d{3}-\d{3}-\d{3}\s?\d{2}\b|\b\d{11}\b")
INN10_RE = re.compile(r"(?<!\d)\d{10}(?!\d)")
INN12_RE = re.compile(r"(?<!\d)\d{12}(?!\d)")
PASSPORT_RE = re.compile(r"(?:(?<!\d)\d{2}\s?\d{2}\s?\d{6}(?!\d))")
CARD_RE = re.compile(r"(?:(?:\d[ -]*?){13,19})")
RS_RE = re.compile(r"(?i)(?:р/с|расч[её]тн(?:ый)?\s+сч[её]т)[^\d]*(\d{20})")
BIK_RE = re.compile(r"(?i)бик[^\d]*(\d{9})")
CVV_RE = re.compile(r"\b(CVV|CVC|CVV2)\b", re.IGNORECASE)

BIOMETRIC_KEYS = [
    'биометр', 'отпечат', 'радуж', 'ирис', 'лицев', 'селфи', 'faceid', 
    'fingerprint', 'iris', 'voiceprint', 'голосов', 'геометрия лица'
]

SPECIAL_KEYS = [
    'диагноз', 'анамнез', 'инвалид', 'здоровь', 'медицин', 'психиатр', 
    'вич', 'религ', 'вероисповед', 'политическ', 'партия', 'интим', 'сексуаль'
]

def count_occurrences(pattern: re.Pattern, text: str) -> int:
    return len(list(pattern.finditer(text)))

def detect_categories(text: str) -> Dict[str, int]:
    t = text if isinstance(text, str) else ''
    low = t.lower()
    cats = {
        'обычные': 0,
        'государственные': 0,
        'платёжные': 0,
        'биометрические': 0,
        'специальные': 0
    }
    
    if not t:
        return cats
    
    cats['обычные'] += count_occurrences(EMAIL_RE, t)
    cats['обычные'] += count_occurrences(PHONE_RE, t)
    cats['обычные'] += min(5, count_occurrences(FIO_RE, t))
    
    for m in DOB_RE.finditer(t):
        if has_context(low, m.start(), 40, 'дата рождения', 'родил', 'д.р.', 'birth'):
            cats['обычные'] += 1
    
    for m in INDEX_RE.finditer(t):
        if has_context(low, m.start(), 40, 'ул', 'улица', 'просп', 'пер', 'дом', 'квартира', 'город', 'г.', 'адрес'):
            cats['обычные'] += 1
    
    for m in SNILS_RE.finditer(t):
        if snils_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN10_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN12_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in PASSPORT_RE.finditer(t):
        if has_context(low, m.start(), 50, 'паспорт', 'серия', 'номер', 'код подразделения'):
            cats['государственные'] += 1
    
    for m in CARD_RE.finditer(t):
        raw = m.group(0)
        digits = re.sub(r'\D', '', raw)
        if 13 <= len(digits) <= 19 and luhn_check(raw):
            if has_context(low, m.start(), 40, 'visa', 'mastercard', 'карта', 'cvv', 'cvc', 'номер карты'):
                cats['платёжные'] += 1
    
    for m in RS_RE.finditer(t):
        cats['платёжные'] += 1
    
    for m in BIK_RE.finditer(t):
        cats['платёжные'] += 1
    
    if CVV_RE.search(t):
        cats['платёжные'] += 1
    
    if any(k in low for k in BIOMETRIC_KEYS):
        cats['биометрические'] += 1
    
    if any(k in low for k in SPECIAL_KEYS):
        cats['специальные'] += 1
    
    return cats

def estimate_uz(cats: Dict[str, int]) -> str:
    total = sum(cats.values())
    distinct = sum(1 for v in cats.values() if v > 0)
    has_special = cats['специальные'] > 0
    has_bio = cats['биометрические'] > 0
    has_pay = cats['платёжные'] > 0
    has_gov = cats['государственные'] > 0
    has_common = cats['обычные'] > 0
    
    if has_special or has_bio:
        return 'УЗ-1' if (total >= 5 or distinct >= 2) else 'УЗ-2'
    if has_pay or has_gov:
        return 'УЗ-2' if (total >= 5 or distinct >= 2) else 'УЗ-3'
    if has_common:
        return 'УЗ-3' if (total >= 5 or distinct >= 2) else 'УЗ-4'
    return 'нет ПДн'

# ============================================================================
# Анализ файла
# ============================================================================

def analyze_file_content(file_path: str, content: bytes) -> Dict[str, Any]:
    result = {
        'path': file_path,
        'success': False,
        'ext': '',
        'text_length': 0,
        'categories': {},
        'total_hits': 0,
        'uz': 'нет ПДн',
        'error': None
    }
    
    if content is None:
        result['error'] = 'Пустое содержимое'
        return result
    
    path = Path(file_path)
    ext = path.suffix.lower().lstrip('.')
    result['ext'] = ext
    
    if ext not in INCLUDE_EXTS:
        result['error'] = f'Неподдерживаемый формат: {ext}'
        return result
    
    try:
        text = extract_text_from_bytes(content, ext)
        result['text_length'] = len(text)
        
        if not text:
            result['error'] = 'Не удалось извлечь текст'
            result['success'] = True 
            return result
        
        cats = detect_categories(text)
        result['categories'] = {k: v for k, v in cats.items() if v > 0}
        result['total_hits'] = sum(cats.values())
        result['uz'] = estimate_uz(cats)
        result['success'] = True
        
    except Exception as e:
        result['error'] = str(e)[:200]
    
    return result

# ============================================================================
# Класс сканера
# ============================================================================

class PDNScanner:
    def __init__(self, max_workers: int = 4):
        self.max_workers = max_workers
        self.results = []
    
    def scan_directory(self, directory_path: str, recursive: bool = True, sample_size: Optional[int] = None) -> pd.DataFrame:
        """Сканирование директории с использованием многопоточности"""
        
        dir_path = Path(directory_path)
        if not dir_path.exists():
            raise FileNotFoundError(f"Директория не найдена: {directory_path}")
        
        print(f"Сбор файлов из: {dir_path.absolute()}")
        
        # Собираем файлы
        files = []
        if recursive:
            for ext in INCLUDE_EXTS:
                pattern = f"*.{ext}"
                files.extend(dir_path.rglob(pattern))
        else:
            for ext in INCLUDE_EXTS:
                pattern = f"*.{ext}"
                files.extend(dir_path.glob(pattern))
        
        files = list(set(files))
        file_paths = [str(f.absolute()) for f in files]
        
        print(f"Найдено файлов: {len(file_paths)}")
        
        if sample_size and sample_size < len(file_paths):
            file_paths = file_paths[:sample_size]
            print(f"Ограничено до: {len(file_paths)} файлов")
        
        if not file_paths:
            print("Нет файлов для анализа")
            return pd.DataFrame()
        
        # Читаем и анализируем файлы с многопоточностью
        print(f"Анализ файлов (потоков: {self.max_workers})...")
        
        results = []
        lock = threading.Lock()
        processed = 0
        
        def process_file(file_path: str):
            nonlocal processed
            try:
                with open(file_path, 'rb') as f:
                    content = f.read()
                result = analyze_file_content(file_path, content)
                with lock:
                    processed += 1
                    if processed % 10 == 0:
                        print(f"  Обработано {processed}/{len(file_paths)} файлов")
                return result
            except Exception as e:
                with lock:
                    processed += 1
                return {
                    'path': file_path,
                    'success': False,
                    'ext': Path(file_path).suffix.lower().lstrip('.'),
                    'text_length': 0,
                    'categories': {},
                    'total_hits': 0,
                    'uz': 'нет ПДн',
                    'error': str(e)[:200]
                }
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {executor.submit(process_file, fp): fp for fp in file_paths}
            for future in as_completed(futures):
                results.append(future.result())
        
        print(f"Анализ завершен. Обработано {len(results)} файлов")
        
        # Конвертируем в DataFrame
        df = pd.DataFrame(results)
        
        # Показываем пример
        if not df.empty:
            print("\nПример результатов:")
            print(df.head(5).to_string())
        
        return df
    
    def get_report_df(self, df: pd.DataFrame) -> pd.DataFrame:
        """Формирование отчета"""
        if df.empty:
            return df
        
        report_df = df.copy()
        report_df['категории_ПДн'] = report_df['categories'].apply(
            lambda x: ', '.join([f"{k}:{v}" for k, v in x.items()]) if x else ''
        )
        report_df = report_df.rename(columns={
            'path': 'путь',
            'total_hits': 'количество_находок',
            'uz': 'УЗ',
            'ext': 'формат_файла'
        })
        
        columns = ['путь', 'категории_ПДн', 'количество_находок', 'УЗ', 'формат_файла']
        return report_df[columns]
    
    def save_report_csv(self, df: pd.DataFrame, output_path: str) -> str:
        """Сохранение отчета в CSV"""
        report_df = self.get_report_df(df)
        
        # Убедимся, что директория существует
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        report_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"CSV отчет сохранен: {output_path}")
        return output_path
    
    def save_report_json(self, df: pd.DataFrame, output_path: str) -> str:
        """Сохранение отчета в JSON"""
        report_df = self.get_report_df(df)
        
        # Конвертируем в JSON-совместимый формат
        json_data = {
            'report_generated': datetime.now().isoformat(),
            'total_files': len(report_df),
            'files_with_pdn': len(report_df[report_df['количество_находок'] > 0]),
            'results': report_df.to_dict(orient='records')
        }
        
        # Добавляем статистику
        stats = self.get_statistics(df)
        json_data['statistics'] = stats
        
        # Убедимся, что директория существует
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)
        
        print(f"JSON отчет сохранен: {output_path}")
        return output_path
    
    def save_report_markdown(self, df: pd.DataFrame, output_path: str) -> str:
        """Сохранение отчета в Markdown с визуализацией"""
        report_df = self.get_report_df(df)
        stats = self.get_statistics(df)
        
        # Убедимся, что директория существует
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        # Формируем Markdown
        md_lines = []
        
        # Заголовок
        md_lines.append("# Отчет по сканированию персональных данных\n")
        md_lines.append(f"**Дата генерации:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        md_lines.append("---\n")
        
        # Общая статистика
        md_lines.append("## 📊 Общая статистика\n")
        md_lines.append("| Показатель | Значение |")
        md_lines.append("|------------|----------|")
        md_lines.append(f"| Всего файлов | {stats['total_files']} |")
        md_lines.append(f"| Успешно обработано | {stats['successful']} |")
        md_lines.append(f"| Ошибок | {stats['failed']} |")
        md_lines.append(f"| **Файлов с ПДн** | **{stats['files_with_pdn']}** |")
        md_lines.append(f"| **Всего находок ПДн** | **{stats['total_pdn_hits']}** |")
        md_lines.append("")
        
        # Распределение по уровням защищенности
        md_lines.append("## 🛡️ Распределение по уровням защищенности (УЗ)\n")
        if stats['protection_levels']:
            md_lines.append("| Уровень защищенности | Количество файлов |")
            md_lines.append("|---------------------|-------------------|")
            for level, count in sorted(stats['protection_levels'].items()):
                md_lines.append(f"| {level} | {count} |")
        else:
            md_lines.append("Файлы с ПДн не найдены")
        md_lines.append("")
        
        # Находки по категориям
        md_lines.append("## 📋 Находки по категориям ПДн\n")
        md_lines.append("| Категория | Количество находок |")
        md_lines.append("|-----------|-------------------|")
        for cat, count in stats['category_totals'].items():
            if count > 0:
                md_lines.append(f"| {cat} | {count} |")
        md_lines.append("")
        
        # Топ-10 файлов с наибольшим количеством находок
        md_lines.append("## 🔍 Топ-10 файлов с наибольшим количеством находок\n")
        top_files = report_df.nlargest(10, 'количество_находок')
        if not top_files.empty:
            md_lines.append("| # | Файл | Категории ПДн | Находок | УЗ | Формат |")
            md_lines.append("|---|------|---------------|---------|-----|--------|")
            for i, row in top_files.iterrows():
                path_short = Path(row['путь']).name
                categories = row['категории_ПДн'][:50] + '...' if len(row['категории_ПДн']) > 50 else row['категории_ПДн']
                md_lines.append(f"| {i+1} | {path_short} | {categories} | {row['количество_находок']} | {row['УЗ']} | {row['формат_файла']} |")
        else:
            md_lines.append("Файлы с ПДн не найдены")
        md_lines.append("")
        
        # Детальный список всех файлов с ПДн
        files_with_pdn = report_df[report_df['количество_находок'] > 0]
        if not files_with_pdn.empty:
            md_lines.append("## 📄 Детальный список файлов с ПДн\n")
            md_lines.append("| Путь к файлу | Категории ПДн | Находок | УЗ | Формат |")
            md_lines.append("|--------------|---------------|---------|-----|--------|")
            for _, row in files_with_pdn.iterrows():
                path_short = Path(row['путь']).name
                md_lines.append(f"| {path_short} | {row['категории_ПДн']} | {row['количество_находок']} | {row['УЗ']} | {row['формат_файла']} |")
        md_lines.append("")
        
        # Визуализация распределения (текстовая диаграмма)
        if stats['protection_levels']:
            md_lines.append("## 📈 Визуализация распределения по УЗ\n")
            md_lines.append("```")
            max_count = max(stats['protection_levels'].values()) if stats['protection_levels'] else 1
            for level, count in sorted(stats['protection_levels'].items()):
                bar_length = int(count / max_count * 40) if max_count > 0 else 0
                bar = "█" * bar_length
                md_lines.append(f"{level:8} | {bar} {count}")
            md_lines.append("```")
        md_lines.append("")
        
        # Визуализация категорий
        cat_with_counts = {k: v for k, v in stats['category_totals'].items() if v > 0}
        if cat_with_counts:
            md_lines.append("## 📊 Визуализация находок по категориям\n")
            md_lines.append("```")
            max_cat = max(cat_with_counts.values()) if cat_with_counts else 1
            for cat, count in cat_with_counts.items():
                bar_length = int(count / max_cat * 40) if max_cat > 0 else 0
                bar = "█" * bar_length
                md_lines.append(f"{cat:12} | {bar} {count}")
            md_lines.append("```")
        
        # Запись в файл
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(md_lines))
        
        print(f"Markdown отчет сохранен: {output_path}")
        return output_path
    
    def save_all_reports(self, df: pd.DataFrame, base_output_path: str) -> Dict[str, str]:
        """Сохранение отчетов во всех форматах"""
        base_path = Path(base_output_path)
        base_name = base_path.stem
        
        reports = {}
        
        # CSV
        csv_path = base_path.parent / f"{base_name}.csv"
        reports['csv'] = self.save_report_csv(df, str(csv_path))
        
        # JSON
        json_path = base_path.parent / f"{base_name}.json"
        reports['json'] = self.save_report_json(df, str(json_path))
        
        # Markdown
        md_path = base_path.parent / f"{base_name}.md"
        reports['markdown'] = self.save_report_markdown(df, str(md_path))
        
        return reports
    
    def get_statistics(self, df: pd.DataFrame) -> Dict[str, Any]:
        """Получение статистики"""
        if df.empty:
            return {
                'total_files': 0,
                'successful': 0,
                'failed': 0,
                'files_with_pdn': 0,
                'protection_levels': {},
                'category_totals': {cat: 0 for cat in PDN_CATEGORIES},
                'total_pdn_hits': 0
            }
        
        total = len(df)
        success = df['success'].sum()
        with_pdn = (df['total_hits'] > 0).sum()
        
        uz_stats = df['uz'].value_counts().to_dict()
        
        cat_totals = {cat: 0 for cat in PDN_CATEGORIES}
        for cats in df['categories']:
            for cat, count in cats.items():
                if cat in cat_totals:
                    cat_totals[cat] += count
        
        return {
            'total_files': total,
            'successful': int(success),
            'failed': total - int(success),
            'files_with_pdn': int(with_pdn),
            'protection_levels': uz_stats,
            'category_totals': cat_totals,
            'total_pdn_hits': sum(cat_totals.values())
        }

# ============================================================================
# Функции для работы с папкой data
# ============================================================================

def scan_data_folder(data_subfolder: str = None, sample_size: int = None, max_workers: int = 4) -> pd.DataFrame:
    """Сканирование папки data"""
    project_root = Path.cwd()
    data_dir = project_root / "data"
    
    if data_subfolder:
        data_dir = data_dir / data_subfolder
    
    if not data_dir.exists():
        raise FileNotFoundError(f"Папка не найдена: {data_dir}")
    
    print(f"Project root: {project_root}")
    print(f"Data directory: {data_dir}")
    
    scanner = PDNScanner(max_workers=max_workers)
    
    results = scanner.scan_directory(
        directory_path=str(data_dir),
        recursive=True,
        sample_size=sample_size
    )
    
    stats = scanner.get_statistics(results)
    
    print("\n" + "="*60)
    print("СТАТИСТИКА СКАНИРОВАНИЯ")
    print("="*60)
    print(f"Всего файлов: {stats['total_files']}")
    print(f"Успешно обработано: {stats['successful']}")
    print(f"Ошибок: {stats['failed']}")
    print(f"Файлов с ПДн: {stats['files_with_pdn']}")
    print(f"Всего находок: {stats['total_pdn_hits']}")
    
    print(f"\nПо уровням защищенности:")
    for level, count in stats['protection_levels'].items():
        print(f"  {level}: {count}")
    
    print(f"\nНаходки по категориям:")
    for cat, count in stats['category_totals'].items():
        if count > 0:
            print(f"  - {cat}: {count}")
    
    return results

def save_report_to_data_folder(df: pd.DataFrame, filename: str = None, formats: List[str] = None) -> Dict[str, Path]:
    """Сохранение отчетов в папку data/reports в указанных форматах"""
    project_root = Path.cwd()
    reports_dir = project_root / "data" / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)
    
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"pdn_report_{timestamp}"
    
    # Убираем расширение если оно есть
    base_name = Path(filename).stem
    base_path = reports_dir / base_name
    
    scanner = PDNScanner()
    
    # Сохраняем в запрошенных форматах
    if formats is None:
        formats = ['csv', 'json', 'markdown']
    
    reports = {}
    
    if 'csv' in formats:
        csv_path = base_path.with_suffix('.csv')
        reports['csv'] = Path(scanner.save_report_csv(df, str(csv_path)))
    
    if 'json' in formats:
        json_path = base_path.with_suffix('.json')
        reports['json'] = Path(scanner.save_report_json(df, str(json_path)))
    
    if 'markdown' in formats or 'md' in formats:
        md_path = base_path.with_suffix('.md')
        reports['markdown'] = Path(scanner.save_report_markdown(df, str(md_path)))
    
    print(f"\n✅ Отчеты сохранены в: {reports_dir}")
    return reports

# ============================================================================
# Основная функция
# ============================================================================

def main():
    parser = argparse.ArgumentParser(description="Сканер ПДн")
    parser.add_argument("--input", "-i", help="Директория для сканирования (по умолчанию ./data)")
    parser.add_argument("--output", "-o", help="Базовое имя файла отчета (без расширения)")
    parser.add_argument("--sample", "-s", type=int, help="Ограничение количества файлов")
    parser.add_argument("--no-recursive", action="store_true", help="Отключить рекурсивный обход")
    parser.add_argument("--workers", "-w", type=int, default=4, help="Количество потоков (по умолчанию 4)")
    parser.add_argument("--format", "-f", choices=['csv', 'json', 'markdown', 'all'], 
                       default='all', help="Формат отчета (по умолчанию all)")
    
    args = parser.parse_args()
    
    if args.input is None:
        args.input = str(Path.cwd() / "data")
    
    if args.output is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        args.output = f"pdn_report_{timestamp}"
    
    formats = ['csv', 'json', 'markdown'] if args.format == 'all' else [args.format]
    
    print(f"Запуск сканирования: {args.input}")
    print(f"Расширения: {', '.join(sorted(INCLUDE_EXTS))}")
    
    scanner = PDNScanner(max_workers=args.workers)
    
    results = scanner.scan_directory(
        directory_path=args.input,
        recursive=not args.no_recursive,
        sample_size=args.sample
    )
    
    # Сохраняем отчеты
    reports_dir = Path(args.input) / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)
    
    base_path = reports_dir / args.output
    
    if 'csv' in formats:
        scanner.save_report_csv(results, str(base_path.with_suffix('.csv')))
    if 'json' in formats:
        scanner.save_report_json(results, str(base_path.with_suffix('.json')))
    if 'markdown' in formats:
        scanner.save_report_markdown(results, str(base_path.with_suffix('.md')))
    
    stats = scanner.get_statistics(results)
    
    print(f"\n📊 ИТОГОВАЯ СТАТИСТИКА:")
    print(f"  Всего файлов: {stats['total_files']}")
   

In [ ]:
# ============================================================================
# Точка входа
# ============================================================================

if __name__ == "__main__":
    # Создаем тестовую папку если её нет
    data_dir = Path.cwd() / "data"
    if not data_dir.exists():
        data_dir.mkdir()
    # Запускаем сканирование
    try:
        # Сканируем папку data
        result_df = scan_data_folder(sample_size=50, max_workers=8)
        
        if result_df is not None and not result_df.empty:
            # Сохраняем отчеты во всех форматах
            reports = save_report_to_data_folder(
                result_df, 
                formats=['csv', 'json', 'markdown']
            )
            
            print("\n" + "="*60)
            print("ФАЙЛЫ С ПЕРСОНАЛЬНЫМИ ДАННЫМИ:")
            print("="*60)
            files_with_pdn = result_df[result_df['total_hits'] > 0]
            if not files_with_pdn.empty:
                print(files_with_pdn[['path', 'categories', 'total_hits', 'uz']].to_string())
            else:
                print("Файлы с ПДн не найдены")
                
            print("\n" + "="*60)
            print("СОХРАНЕННЫЕ ОТЧЕТЫ:")
            print("="*60)
            for fmt, path in reports.items():
                print(f"  {fmt.upper()}: {path}")
        else:
            print("Нет результатов для отображения")
            
    except Exception as e:
        print(f"Ошибка: {e}")
        import traceback
        traceback.print_exc()

Project root: c:\Users\rulan\Downloads\SamsungHacaton
Data directory: c:\Users\rulan\Downloads\SamsungHacaton\data
Сбор файлов из: c:\Users\rulan\Downloads\SamsungHacaton\data
Найдено файлов: 3307
Ограничено до: 50 файлов
Анализ файлов (потоков: 8)...
  Обработано 10/50 файлов
  Обработано 20/50 файлов
  Обработано 30/50 файлов
  Обработано 40/50 файлов
  Обработано 50/50 файлов
Анализ завершен. Обработано 50 файлов

Пример результатов:
                                                                                        path  success  ext  text_length      categories  total_hits       uz                     error
0         c:\Users\rulan\Downloads\SamsungHacaton\data\share\Прочее\katalog_app_magistri.pdf     True  pdf            0              {}           0  нет ПДн  Не удалось извлечь текст
1  c:\Users\rulan\Downloads\SamsungHacaton\data\share\Прочее\Program_Priority_directions.pdf     True  pdf            0              {}           0  нет ПДн  Не удалось извлечь текст
2         